# AzLegalRAG - Colab Demo

Azerbaijani Legal Q&A using RAG with bge-m3 embeddings.

**Requirements**: T4/L4/A100 GPU

Go to: Runtime > Change runtime type > GPU

## 1. Install Dependencies

In [ ]:
!pip install -q torch transformers accelerate sentence-transformers langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface chromadb datasets tqdm huggingface_hub

## 2. HuggingFace Login (Required for Gated Dataset)

The `allmalab/eqanun` dataset is gated. You need to:
1. Create a token at https://huggingface.co/settings/tokens
2. Accept access at https://huggingface.co/datasets/allmalab/eqanun
3. Run the cell below and paste your token

In [ ]:
from huggingface_hub import login
login()

## 3. Clone Repository

In [ ]:
!git clone https://github.com/StartZer0/AzLegalRAG.git
%cd AzLegalRAG

## 4. Check GPU & Setup Path

In [ ]:
import sys
sys.path.insert(0, './src')

import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 5a. OPTION A: Load Vectorstore from Google Drive (If Already Built)

**Skip to 5b if building for the first time.**

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Check if vectorstore exists in Drive
import os
drive_path = '/content/drive/MyDrive/AzLegalRAG/vectorstore'
local_path = './vectorstore'

if os.path.exists(drive_path):
    print(f"Found vectorstore in Drive! Copying to local...")
    !cp -r {drive_path} {local_path}
    print("Done! You can skip cell 5b.")
else:
    print("No vectorstore found in Drive. Run cell 5b to create one.")

## 5b. OPTION B: Build Vectorstore (First Time - ~30 min on L4)

**Note**: Text is automatically normalized during ingestion to fix split Azerbaijani characters.

In [ ]:
from ingest import load_eqanun, chunk_documents

# Load dataset
dataset = load_eqanun()
print(f"Loaded {len(dataset)} documents")

# Chunk documents (normalize=True cleans Azerbaijani text)
chunks = chunk_documents(dataset, normalize=True)
print(f"Created {len(chunks)} chunks")

In [ ]:
# Create vectorstore with bge-m3 embeddings
from embed import create_vectorstore

vectorstore = create_vectorstore(chunks)
print("Vectorstore created!")

## 5c. Save Vectorstore to Google Drive (RUN THIS AFTER BUILDING!)

**IMPORTANT: Run this to save your vectorstore permanently.**

In [ ]:
from google.colab import drive
import os

# Mount if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# Save vectorstore to Drive
drive_path = '/content/drive/MyDrive/AzLegalRAG'
!mkdir -p {drive_path}
!cp -r ./vectorstore {drive_path}/

print(f"✅ Vectorstore saved to Google Drive: {drive_path}/vectorstore")
print("Your 465K embeddings are now safe!")

## 6. Test Search

In [ ]:
from embed import load_vectorstore
from retrieve import semantic_search

vs = load_vectorstore()
results = semantic_search(vs, "Emek muqavilesi nedir?", k=3)

for i, doc in enumerate(results, 1):
    print(f"\n[{i}] Source: {doc.metadata['source']}")
    print(doc.page_content[:300] + "...")

## 7. Load LLM and Create RAG Chain

In [ ]:
from generate import get_llm, create_rag_chain
from embed import load_vectorstore

vs = load_vectorstore()
llm = get_llm()
chain, retriever = create_rag_chain(vs, llm)
print("RAG chain ready!")

## 8. Test RAG Q&A

In [ ]:
# Simple Q&A using the chain
question = "Kollektiv müqavilə nə qədər müddətə bağlanıla bilər?"
answer = chain.invoke(question)

print("=" * 50)
print(f"Question: {question}")
print("=" * 50)
print(f"\nAnswer:\n{answer}")

In [ ]:
# Try more questions
questions = [
    "Əmək müqaviləsi nədir?",
    "Mehkeme qerari nece shekillendirilir?",
    "Nikah muqavilesi ucun ne teleb olunur?"
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    answer = chain.invoke(q)
    print(f"A: {answer[:500]}...")